In [16]:
import pandas as pd

In [17]:
datos = pd.read_csv("xCafeina.csv")

In [18]:
datos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 935 entries, 0 to 934
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  935 non-null    int64  
 1        Angle  935 non-null    float64
 2          PSD  935 non-null    int64  
dtypes: float64(1), int64(2)
memory usage: 22.0 KB


In [19]:
datos.head()

,Unnamed: 0,Angle,PSD
0,0,5.00000,2061
1,1,5.04003,2045
2,2,5.08006,2050
3,3,5.12010,2000
4,4,5.16013,2009


In [20]:



csv_file     = "xCafeina.csv"                  # archivo de datos
sample_label = "Cafeína tableta"               # nombre que saldrá en títulos
html_output  = "xCafeina_difractogramas.html"  # nombre del HTML

# ======= 0. Config común para PNGs en alta resolución =======
def make_config(nombre_png):
    return dict(
        displaylogo=False,
        toImageButtonOptions=dict(
            format="png",
            filename=nombre_png,
            height=1200,   # más alto
            width=2000,    # más ancho
            scale=3        # más resolución (3x)
        )
    )

# ======= 1. Leer y preparar datos =======
datos = pd.read_csv(csv_file)
df = datos.copy()
df.columns = df.columns.str.strip()
if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

# ======= 2. Función para figura de región =======
def make_region_fig(df, tmin, tmax, titulo_extra=""):
    mask = (df["Angle"] >= tmin) & (df["Angle"] <= tmax)
    df_zoom = df.loc[mask].copy()

    titulo_base = f"Región {tmin:.0f}°–{tmax:.0f}°"
    titulo = f"{titulo_base} – {titulo_extra}" if titulo_extra else titulo_base

    fig = px.line(
        df_zoom,
        x="Angle",
        y="PSD",
        title=titulo,
        labels={"Angle": "2θ (°)", "PSD": "Intensidad (cuentas PSD)"}
    )

    fig.update_traces(hovertemplate="2θ = %{x:.2f}°<br>I = %{y:.0f}")

    fig.update_layout(
        width=900,
        height=400,
        margin=dict(l=40, r=40, t=60, b=40)
    )

    return fig

# ======= 3. Figura TOTAL =======
fig_total = px.line(
    df,
    x="Angle",
    y="PSD",
    title=f"Difractograma de rayos X – {sample_label} (interactivo)",
    labels={"Angle": "2θ (°)", "PSD": "Intensidad (cuentas PSD)"}
)
fig_total.update_traces(hovertemplate="2θ = %{x:.2f}°<br>I = %{y:.0f}")
fig_total.update_xaxes(dtick=5)
fig_total.update_layout(
    width=900,
    height=500,
    margin=dict(l=40, r=40, t=60, b=40)
)

# ======= 4. Figuras de regiones =======
fig_r1 = make_region_fig(df, 5, 18,  "pico dominante")
fig_r2 = make_region_fig(df, 20, 35, "picos secundarios")
fig_r3 = make_region_fig(df, 35, 45, "cola de difracción")

# ======= 5. Convertir figuras a HTML embebido (con config HD) =======
html_total = pio.to_html(
    fig_total,
    full_html=False,
    include_plotlyjs="cdn",
    config=make_config("xCafeina_total")
)

html_r1 = pio.to_html(
    fig_r1,
    full_html=False,
    include_plotlyjs=False,
    config=make_config("xCafeina_region_5_18")
)

html_r2 = pio.to_html(
    fig_r2,
    full_html=False,
    include_plotlyjs=False,
    config=make_config("xCafeina_region_20_35")
)

html_r3 = pio.to_html(
    fig_r3,
    full_html=False,
    include_plotlyjs=False,
    config=make_config("xCafeina_region_35_45")
)

# ======= 6. Construir la página HTML completa (con CSS) =======
html_page = f"""
<!DOCTYPE html>
<html lang="es">
<head>
  <meta charset="UTF-8">
  <title>Difractogramas de rayos X – {sample_label}</title>
  <style>
    body {{
      margin: 0;
      padding: 2rem;
      font-family: system-ui, -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
      background: #020617;
      color: #e5e7eb;
    }}
    h1 {{
      text-align: center;
      margin-bottom: 2rem;
    }}
    h2 {{
      margin-top: 0;
      text-align: center;
      color: #e5e7eb;
    }}
    .section {{
      background: #0f172a;
      border-radius: 16px;
      padding: 1.5rem;
      margin: 0 auto 2rem auto;
      max-width: 1100px;
      box-shadow: 0 18px 45px rgba(0,0,0,0.5);
    }}
  </style>
</head>
<body>
  <h1>Difractogramas de rayos X – {sample_label}</h1>

  <div class="section">
    <h2>Difractograma completo</h2>
    {html_total}
  </div>

  <div class="section">
    <h2>Región 5°–18° – pico dominante</h2>
    {html_r1}
  </div>

  <div class="section">
    <h2>Región 20°–35° – picos secundarios</h2>
    {html_r2}
  </div>

  <div class="section">
    <h2>Región 35°–45° – cola de difracción</h2>
    {html_r3}
  </div>
</body>
</html>
"""

# ======= 7. Guardar y abrir en el navegador =======
out_path = pathlib.Path(html_output)
out_path.write_text(html_page, encoding="utf-8")

webbrowser.open(out_path.resolve().as_uri())
print(f"Página generada: {out_path}")




Página generada: xCafeina_difractogramas.html
